### Paso 1 — Carga del panel de ventanas electorales

Cargamos `panel_ventanas.csv` 

**Esperado**: `municipal: 12x127`, `provincial: 12x127`, `nacional: 7x127`

In [11]:
import pandas as pd
import sys
general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")  
from ml_models.cargar_panel import cargar_panel 

NIVELES = ["municipal", "provincial", "nacional"]
paneles = {nivel: cargar_panel(nivel,f"{data_path}panel_ventanas.csv") for nivel in NIVELES}

for nivel, df in paneles.items():
    print(f"{nivel}: {df.shape[0]} filas x {df.shape[1]} columnas")

municipal: 12 filas x 127 columnas
provincial: 12 filas x 127 columnas
nacional: 7 filas x 127 columnas


### Paso 2 — Revisión de variables (matriz de correlación)

El objetivo es visualizar variables redundantes

In [14]:
nivel = "municipal" 
df = paneles[nivel]

cols_vc = [
    c for c in df.columns
    if c.endswith("_vc") and pd.api.types.is_numeric_dtype(df[c])
]

corr = df[cols_vc].corr(method="pearson")  # pairwise, ignora NaN automáticamente

print(f"{nivel}: {len(cols_vc)} variables _vc, N={len(df)}")
corr

municipal: 37 variables _vc, N=12


,desocupacion_final_vc,desocupacion_nivel_vc,desocupacion_pendiente_vc,desocupacion_volatilidad_vc,emae_final_vc,emae_nivel_vc,emae_pendiente_vc,emae_volatilidad_vc,icc_final_vc,icc_nivel_vc,...,resultado_fiscal_pendiente_vc,resultado_fiscal_volatilidad_vc,salario_real_final_vc,salario_real_nivel_vc,salario_real_pendiente_vc,salario_real_volatilidad_vc,tc_oficial_final_vc,tc_oficial_nivel_vc,tc_oficial_pendiente_vc,tc_oficial_volatilidad_vc
desocupacion_final_vc,1.000000,0.967016,-0.892786,0.485517,-0.740469,-0.704289,-0.067340,0.035119,-0.367137,-0.418511,...,0.038489,-0.227004,-0.388741,-0.308488,-0.273736,-0.141247,-0.219954,-0.209513,-0.233651,-0.231804
desocupacion_nivel_vc,0.967016,1.000000,-0.919701,0.663285,-0.787546,-0.852487,0.264270,0.132324,-0.285541,-0.301427,...,0.017468,-0.233228,-0.424350,-0.345234,-0.260197,-0.253594,-0.266020,-0.255274,-0.281650,-0.279971
desocupacion_pendiente_vc,-0.892786,-0.919701,1.000000,-0.456562,0.531637,0.726332,-0.688258,-0.328348,0.188713,0.327325,...,0.109835,0.341397,0.440421,0.399307,0.119471,0.375330,0.210948,0.207726,0.213888,0.213485
desocupacion_volatilidad_vc,0.485517,0.663285,-0.456562,1.000000,-0.649763,-0.754252,0.389727,0.063731,-0.146715,0.037211,...,0.229329,0.118886,-0.178111,-0.107868,-0.246661,-0.142980,-0.138704,-0.125492,-0.164768,-0.163047
emae_final_vc,-0.740469,-0.787546,0.531637,-0.649763,1.000000,0.937717,-0.022200,-0.139702,0.083070,-0.380826,...,0.266516,0.360963,0.628997,0.524213,0.139652,0.445185,0.339913,0.333188,0.349312,0.348770
emae_nivel_vc,-0.704289,-0.852487,0.726332,-0.754252,0.937717,1.000000,-0.358049,-0.331488,-0.156203,-0.593617,...,0.317018,0.417480,0.682740,0.594128,0.016899,0.489619,0.324931,0.311798,0.347100,0.345560
emae_pendiente_vc,-0.067340,0.264270,-0.688258,0.389727,-0.022200,-0.358049,1.000000,0.524923,0.590606,0.645797,...,-0.257539,-0.224941,-0.312088,-0.327862,0.446168,-0.229857,0.094309,0.110495,0.060493,0.062829
emae_volatilidad_vc,0.035119,0.132324,-0.328348,0.063731,-0.139702,-0.331488,0.524923,1.000000,0.632677,0.638326,...,-0.452748,-0.500234,-0.512624,-0.522912,0.092627,-0.476295,-0.502385,-0.481152,-0.538529,-0.535687
icc_final_vc,-0.367137,-0.285541,0.188713,-0.146715,0.083070,-0.156203,0.590606,0.632677,1.000000,0.755668,...,-0.430499,-0.543213,-0.346358,-0.395716,0.262404,-0.345966,-0.197192,-0.189837,-0.208397,-0.207307
icc_nivel_vc,-0.418511,-0.301427,0.327325,0.037211,-0.380826,-0.593617,0.645797,0.638326,0.755668,1.000000,...,-0.561208,-0.498836,-0.398036,-0.431806,0.409915,-0.401327,-0.235578,-0.220806,-0.261492,-0.259385


### Paso 3 — Sub-selección: colapsar clusters redundantes

La matriz de correlación mostró un cluster de colinealidad casi perfecta entre `ipc_*` y `tc_oficial_*` (r > 0.98 en todo el bloque), además de pares menores en `desocupacion`, `resultado_fiscal` y `salario_real`. Se busca simplificar reduciendo los datos que son redundantes para evitar que LASSO elija de forma aleatoria entre esas opciones.

**Parametrización fijada:**

| Decisión | Valor |
|---|---|
| Umbral de redundancia | `\|r\| ≥ 0.90` |
| Alcance | Transitivo (single-linkage): si A-B≥0.90 y B-C≥0.90, A/B/C van al mismo cluster aunque A-C no llegue al umbral |
| Desempate 1 | Sufijo `_nivel_vc` preferido sobre `_final`/`_pendiente`/`_volatilidad`/`_acum` |
| Desempate 2 | Prioridad teórica de la variable (`ipc` > `desocupacion` > `icg` > `icc` > `salario_real` > `tc_oficial` > `reservas` > `resultado_fiscal`) — orden de relevancia en la literatura de voto económico citada, no un criterio estadístico |

In [ ]:
UMBRAL_REDUNDANCIA = 0.90


def encontrar_redundantes(corr: pd.DataFrame, umbral: float) -> list[set[str]]:
    """Agrupa columnas en clusters de redundancia transitiva (single-linkage):
    si A-B >= umbral y B-C >= umbral, A/B/C quedan juntas aunque A-C no
    supere el umbral directamente."""
    cols = list(corr.columns)
    adyacencia = {c: set() for c in cols}
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            if abs(corr.loc[a, b]) >= umbral:
                adyacencia[a].add(b)
                adyacencia[b].add(a)

    visit, clusters = set(), []
    for c in cols:
        if c in visit:
            continue
        stack, cluster = [c], set()
        while stack:
            actual = stack.pop()
            if actual in cluster:
                continue
            cluster.add(actual)
            stack.extend(adyacencia[actual] - cluster)
        visit |= cluster
        clusters.append(cluster)
    return clusters


def elegir_representante(cluster: set[str], df: pd.DataFrame) -> str:
    """Del cluster, la columna con menos NaN; empate -> preferencia de sufijo."""
    orden_sufijo = ["_nivel_vc", "_final_vc", "_pendiente_vc", "_volatilidad_vc", "_acum_vc"]

    def clave(col):
        n_nan = df[col].isna().sum()
        pref = next((i for i, s in enumerate(orden_sufijo) if col.endswith(s)), 99)
        return (n_nan, pref)

    return min(cluster, key=clave)


clusters = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
cols_finales = [elegir_representante(cl, df) if len(cl) > 1 else next(iter(cl)) for cl in clusters]

print(f"De {len(cols_vc)} columnas _vc, quedan {len(cols_finales)} tras colapsar clusters (umbral={UMBRAL_REDUNDANCIA})\n")
for cl in clusters:
    if len(cl) > 1:
        elegido = elegir_representante(cl, df)
        print(f"cluster ({len(cl)}): {sorted(cl)} -> queda: {elegido}")

De 37 columnas _vc, quedan 24 tras colapsar clusters (umbral=0.9)

cluster (3): ['desocupacion_final_vc', 'desocupacion_nivel_vc', 'desocupacion_pendiente_vc'] -> queda: desocupacion_nivel_vc
cluster (2): ['emae_final_vc', 'emae_nivel_vc'] -> queda: emae_nivel_vc
cluster (8): ['ipc_final_vc', 'ipc_nivel_vc', 'ipc_pendiente_vc', 'ipc_volatilidad_vc', 'tc_oficial_final_vc', 'tc_oficial_nivel_vc', 'tc_oficial_pendiente_vc', 'tc_oficial_volatilidad_vc'] -> queda: tc_oficial_nivel_vc
cluster (2): ['resultado_fiscal_final_vc', 'resultado_fiscal_nivel_vc'] -> queda: resultado_fiscal_nivel_vc
cluster (3): ['resultado_fiscal_volatilidad_vc', 'salario_real_final_vc', 'salario_real_nivel_vc'] -> queda: salario_real_nivel_vc


In [18]:
PRIORIDAD_TEORICA = ["ipc", "desocupacion", "icg", "icc", "salario_real", "tc_oficial", "reservas", "resultado_fiscal", "emae"]
ORDEN_SUFIJO = ["_nivel_vc", "_final_vc", "_pendiente_vc", "_volatilidad_vc", "_acum_vc"]


def elegir_representante(cluster: set[str]) -> str:
    """Del cluster: prioridad de sufijo primero, después prioridad
    teórica de la variable; empate final -> orden alfabético
    (determinístico, nunca el orden de un set)."""

    def clave(col):
        pref_sufijo = next((i for i, s in enumerate(ORDEN_SUFIJO) if col.endswith(s)), 99)
        variable = next((v for v in PRIORIDAD_TEORICA if col.startswith(v + "_")), None)
        pref_teorica = PRIORIDAD_TEORICA.index(variable) if variable else 99
        return (pref_sufijo, pref_teorica, col)

    return min(cluster, key=clave)

In [19]:
clusters = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
cols_finales = [elegir_representante(cl) if len(cl) > 1 else next(iter(cl)) for cl in clusters]

print(f"De {len(cols_vc)} columnas _vc, quedan {len(cols_finales)} tras colapsar clusters (umbral={UMBRAL_REDUNDANCIA})\n")
for cl in clusters:
    if len(cl) > 1:
        elegido = elegir_representante(cl)
        print(f"cluster ({len(cl)}): {sorted(cl)} -> queda: {elegido}")

De 37 columnas _vc, quedan 24 tras colapsar clusters (umbral=0.9)

cluster (3): ['desocupacion_final_vc', 'desocupacion_nivel_vc', 'desocupacion_pendiente_vc'] -> queda: desocupacion_nivel_vc
cluster (2): ['emae_final_vc', 'emae_nivel_vc'] -> queda: emae_nivel_vc
cluster (8): ['ipc_final_vc', 'ipc_nivel_vc', 'ipc_pendiente_vc', 'ipc_volatilidad_vc', 'tc_oficial_final_vc', 'tc_oficial_nivel_vc', 'tc_oficial_pendiente_vc', 'tc_oficial_volatilidad_vc'] -> queda: ipc_nivel_vc
cluster (2): ['resultado_fiscal_final_vc', 'resultado_fiscal_nivel_vc'] -> queda: resultado_fiscal_nivel_vc
cluster (3): ['resultado_fiscal_volatilidad_vc', 'salario_real_final_vc', 'salario_real_nivel_vc'] -> queda: salario_real_nivel_vc


In [20]:
cols_finales_municipal = cols_finales  # el resultado que ya tenés (24 columnas)

def construir_Xy_final(nivel: str, columnas: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    df = paneles[nivel]
    X = df[columnas].copy()
    y = df["delta_v"].copy()

    completas = X.notna().all(axis=1) & y.notna()
    n_incompletas = (~completas).sum()
    if n_incompletas:
        print(f"[{nivel}] excluye {n_incompletas} fila(s) por NaN: {df.loc[~completas, 'id_transicion'].tolist()}")

    return X.loc[completas].reset_index(drop=True), y.loc[completas].reset_index(drop=True)


datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, cols_finales_municipal)
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
municipal: N=11, P=24
[provincial] excluye 1 fila(s) por NaN: ['provincial_2001_2003']
provincial: N=11, P=24
nacional: N=7, P=24


In [22]:
def columnas_nan(nivel: str, id_transicion: str, columnas: list[str]) -> list[str]:
    """Devuelve las columnas de `columnas` que son NaN para esa fila puntual."""
    df = paneles[nivel]
    fila = df[df["id_transicion"] == id_transicion]
    if fila.empty:
        raise ValueError(f"no encontré {id_transicion!r} en el panel de {nivel}")
    fila = fila.iloc[0]
    return [c for c in columnas if pd.isna(fila[c])]


for nivel, id_t in [("municipal", "municipal_2001_2003"), ("provincial", "provincial_2001_2003")]:
    faltantes = columnas_nan(nivel, id_t, cols_finales_municipal)
    print(f"{id_t}: NaN en -> {faltantes}")

municipal_2001_2003: NaN en -> ['emae_pendiente_vc', 'emae_volatilidad_vc']
provincial_2001_2003: NaN en -> ['emae_pendiente_vc', 'emae_volatilidad_vc']


In [24]:
cols_finales_municipal = [c for c in cols_finales_municipal if not c.startswith("emae_")]
 

datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, cols_finales_municipal)
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

municipal: N=12, P=21
provincial: N=12, P=21
nacional: N=7, P=21


### Paso 4 — LASSO por coordinate descent (implementación propia)

Formulación: `(1/2n)·‖y - Xβ‖² + α·‖β‖₁` (convención sklearn/glmnet). `X` estandarizada a mano (`ddof=0`), `y` centrada; intercepto = `media(y)`, no se penaliza.

`soft_threshold`: operador proximal de L1, da la selección de variables (coeficiente exactamente en cero si `|z| <= alpha`). `lasso_coordinate_descent`: actualiza una coordenada de `β` a la vez hasta que el cambio máximo entre iteraciones sea `< tol`.

In [ ]:
import numpy as np

def estandarizar(X: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    medias = X.mean(axis=0).values
    desvios = X.std(axis=0, ddof=0).values
    X_std = (X.values - medias) / desvios
    return X_std, medias, desvios


def soft_threshold(z: float, umbral: float) -> float:
    """Operador proximal de la norma L1: encoge z hacia cero, y lo pone en cero
    exactamente si |z| <= umbral."""
    if z > umbral:
        return z - umbral
    elif z < -umbral:
        return z + umbral
    return 0.0


def lasso_coordinate_descent(
    X: np.ndarray, y: np.ndarray, alpha: float, tol: float = 1e-6, max_iter: int = 1000
) -> np.ndarray:
    n, p = X.shape
    beta = np.zeros(p)
    for iteracion in range(max_iter):
        beta_anterior = beta.copy()
        for j in range(p):
            residuo_parcial = y - X @ beta + X[:, j] * beta[j]  
            rho_j = X[:, j] @ residuo_parcial / n
            beta[j] = soft_threshold(rho_j, alpha)

        if np.max(np.abs(beta - beta_anterior)) < tol:
            break

    return beta

**Validación 1 — condiciones KKT.** Para `β_j != 0`: `Xⱼᵀ(y-Xβ)/n = α·signo(β_j)`. Para `β_j = 0`: `|Xⱼᵀ(y-Xβ)/n| <= α`. Con `alpha=1`, error máximo en activos `~5e-7` -- converge correctamente.

In [ ]:
def verificar_kkt(X: np.ndarray, y: np.ndarray, beta: np.ndarray, alpha: float, n: int) -> dict:
    gradiente = X.T @ (y - X @ beta) / n  # Xⱼᵀ(y - Xβ)/n para cada j

    activos = beta != 0
    inactivos = ~activos

    # para los activos: gradiente debería ser ~alpha * signo(beta)
    error_activos = np.max(np.abs(gradiente[activos] - alpha * np.sign(beta[activos]))) if activos.any() else 0.0

    # para los inactivos: |gradiente| debería ser <= alpha
    exceso_inactivos = np.max(np.abs(gradiente[inactivos]) - alpha) if inactivos.any() else -np.inf

    return {
        "error_max_en_activos": error_activos,
        "exceso_max_en_inactivos": exceso_inactivos,  
        "n_activos": activos.sum(),
    }


nivel = "municipal"
X_df, y_ser = datos_final[nivel]
X_std, medias, desvios = estandarizar(X_df)
y_centrado = y_ser.values - y_ser.mean()
n = len(y_centrado)

alpha_prueba = 1.0
beta_manual = lasso_coordinate_descent(X_std, y_centrado, alpha_prueba)

kkt = verificar_kkt(X_std, y_centrado, beta_manual, alpha_prueba, n)
print("KKT:", kkt)

# Chequeo 2: alpha=0 debe coincidir con OLS
beta_alpha_cero = lasso_coordinate_descent(X_std, y_centrado, alpha=0.0, max_iter=5000)
beta_ols, *_ = np.linalg.lstsq(X_std, y_centrado, rcond=None)
print("Máxima diferencia vs. OLS (alpha=0):", np.max(np.abs(beta_alpha_cero - beta_ols)))

KKT: {'error_max_en_activos': np.float64(5.235086024679703e-07), 'exceso_max_en_inactivos': np.float64(-0.10473658184170365), 'n_activos': np.int64(8)}
Máxima diferencia vs. OLS (alpha=0): 3.391581537103192


**Validación 2 — `alpha=0` vs. OLS (`np.linalg.lstsq`).** Diferencia grande (3.39) entre los `β`, pero **no es bug**: con P=21 > N-1, el sistema sin penalizar es subdeterminado (infinitas soluciones con error de entrenamiento ~0). Confirmado: ambos residuos (`manual` y `OLS`) dan `~0` (`4.9e-6` y `4e-14`) -- son dos soluciones distintas y válidas del mismo sistema indeterminado, no una implementación distinta a la otra.

In [28]:
residuo_manual = y_centrado - X_std @ beta_alpha_cero
residuo_ols = y_centrado - X_std @ beta_ols

print("Residuo manual (debería ser ~0 si el sistema es subdeterminado):", np.max(np.abs(residuo_manual)))
print("Residuo OLS (debería ser ~0 también):", np.max(np.abs(residuo_ols)))

Residuo manual (debería ser ~0 si el sistema es subdeterminado): 4.891921480343342e-06
Residuo OLS (debería ser ~0 también): 3.9745984281580604e-14


**Validación 3 — contra `sklearn.linear_model.Lasso`**, mismo `alpha`, `fit_intercept=False` (y ya viene centrado). Diferencia máxima decrece con `alpha` (`1.04e-06` en `alpha=0.1` → `0.00` en `alpha=20.0`): a mayor regularización el problema deja de ser subdeterminado y ambas implementaciones convergen al mismo óptimo único. Confirma la implementación propia sin reemplazarla.

In [29]:
from sklearn.linear_model import Lasso

def validar_contra_sklearn(X_std: np.ndarray, y_centrado: np.ndarray, alpha: float) -> float:
    """Diferencia máxima entre coordinate descent propio y sklearn, para
    el mismo alpha. fit_intercept=False porque y ya viene centrado --
    sklearn no debe volver a centrar ni ajustar un intercepto propio."""
    beta_manual = lasso_coordinate_descent(X_std, y_centrado, alpha, tol=1e-8, max_iter=5000)

    modelo_ref = Lasso(alpha=alpha, fit_intercept=False, max_iter=10000, tol=1e-8).fit(X_std, y_centrado)
    beta_ref = modelo_ref.coef_

    return np.max(np.abs(beta_manual - beta_ref))


nivel = "municipal"
X_df, y_ser = datos_final[nivel]
X_std, medias, desvios = estandarizar(X_df)
y_centrado = y_ser.values - y_ser.mean()

for alpha_prueba in [0.1, 1.0, 5.0, 20.0]:
    diff = validar_contra_sklearn(X_std, y_centrado, alpha_prueba)
    print(f"alpha={alpha_prueba}: diferencia máxima vs. sklearn = {diff:.2e}")

alpha=0.1: diferencia máxima vs. sklearn = 1.04e-06
alpha=1.0: diferencia máxima vs. sklearn = 1.02e-07
alpha=5.0: diferencia máxima vs. sklearn = 1.92e-09
alpha=20.0: diferencia máxima vs. sklearn = 0.00e+00


### Paso 5 — Grilla de alpha + LOO-CV manual

**Resultado inicial** (grilla x1, techo = alpha_max sin ampliar):

| Nivel | alpha_min | alpha_1se | techo grilla |
|---|---|---|---|
| Municipal | 3.92 | 6.90 | 10.53 |
| Provincial | 4.87 | 9.86 | 9.86 (pegado) |
| Nacional | 9.20 | 9.20 | 9.20 (pegado) |

In [31]:
def alpha_max_lasso(X: np.ndarray, y: np.ndarray) -> float:
    """El alpha más chico a partir del cual todos los coeficientes quedan
    en cero (condición KKT en beta=0: |Xⱼᵀy|/n <= alpha para todo j)."""
    n = len(y)
    return np.max(np.abs(X.T @ y)) / n


def lasso_loocv_manual(X_df: pd.DataFrame, y_ser: pd.Series, n_alphas: int = 50) -> dict:
    n = len(y_ser)
    alpha_max = alpha_max_lasso(*estandarizar(X_df)[:1], y_ser.values - y_ser.mean())
    # nota: estandarizar(X_df)[0] ya es X_std; ver llamada real abajo
    X_std_full, medias, desvios = estandarizar(X_df)
    y_full = y_ser.values
    alpha_max = alpha_max_lasso(X_std_full, y_full - y_full.mean())
    alphas = np.logspace(np.log10(alpha_max * 1e-3), np.log10(alpha_max), n_alphas)

    errores = np.zeros((n, n_alphas))

    for i in range(n):
        idx_train = [k for k in range(n) if k != i]
        X_train_df = X_df.iloc[idx_train]
        y_train = y_ser.iloc[idx_train].values
        X_test_df = X_df.iloc[[i]]
        y_test = y_ser.iloc[i]

        X_train_std, medias_tr, desvios_tr = estandarizar(X_train_df)
        y_train_media = y_train.mean()
        y_train_centrado = y_train - y_train_media

        X_test_std = (X_test_df.values - medias_tr) / desvios_tr

        beta = np.zeros(X_train_std.shape[1])  # warm start arranca acá, se actualiza en el loop
        for j, alpha in enumerate(alphas[::-1]):  # de mayor a menor -> warm start natural
            beta = lasso_coordinate_descent(X_train_std, y_train_centrado, alpha, tol=1e-6, max_iter=1000)
            pred = y_train_media + X_test_std @ beta
            errores[i, len(alphas) - 1 - j] = (pred[0] - y_test) ** 2

    mean_mse = errores.mean(axis=0)
    se_mse = errores.std(axis=0, ddof=1) / np.sqrt(n)

    idx_min = mean_mse.argmin()
    umbral = mean_mse[idx_min] + se_mse[idx_min]
    alpha_1se = alphas[mean_mse <= umbral].max()

    return {"alphas": alphas, "mean_mse": mean_mse, "se_mse": se_mse, "alpha_min": alphas[idx_min], "alpha_1se": alpha_1se}

In [35]:
def lasso_loocv_manual(X_df: pd.DataFrame, y_ser: pd.Series, n_alphas: int = 50, factor_extension: float = 1.0) -> dict:
    n = len(y_ser)
    X_std_full, _, _ = estandarizar(X_df)
    y_full = y_ser.values
    alpha_max = alpha_max_lasso(X_std_full, y_full - y_full.mean()) * factor_extension
    alphas = np.logspace(np.log10(alpha_max * 1e-3), np.log10(alpha_max), n_alphas)

    errores = np.zeros((n, n_alphas))

    for i in range(n):
        idx_train = [k for k in range(n) if k != i]
        X_train_df = X_df.iloc[idx_train]
        y_train = y_ser.iloc[idx_train].values
        X_test_df = X_df.iloc[[i]]
        y_test = y_ser.iloc[i]

        X_train_std, medias_tr, desvios_tr = estandarizar(X_train_df)
        y_train_media = y_train.mean()
        y_train_centrado = y_train - y_train_media
        X_test_std = (X_test_df.values - medias_tr) / desvios_tr

        beta = np.zeros(X_train_std.shape[1])
        for j, alpha in enumerate(alphas[::-1]):  # de mayor a menor -> warm start natural
            beta = lasso_coordinate_descent(X_train_std, y_train_centrado, alpha, tol=1e-6, max_iter=1000)
            pred = y_train_media + X_test_std @ beta
            errores[i, len(alphas) - 1 - j] = (pred[0] - y_test) ** 2

    mean_mse = errores.mean(axis=0)
    se_mse = errores.std(axis=0, ddof=1) / np.sqrt(n)

    idx_min = mean_mse.argmin()
    umbral = mean_mse[idx_min] + se_mse[idx_min]
    alpha_1se = alphas[mean_mse <= umbral].max()

    return {"alphas": alphas, "mean_mse": mean_mse, "se_mse": se_mse, "alpha_min": alphas[idx_min], "alpha_1se": alpha_1se}


resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

municipal: alpha_min=3.8109  alpha_1se=7.7118  (techo grilla=31.5791)
provincial: alpha_min=5.4508  alpha_1se=29.5909  (techo grilla=29.5909)
nacional: alpha_min=11.8511  alpha_1se=27.6125  (techo grilla=27.6125)


Provincial y nacional pegan en el techo -- se amplía la grilla (x3, x10) para descartar que sea límite artificial de rango, no saturación real.

**Verificación (grilla ampliada):**

| Nivel | MSE mínimo (x1→x3→x10) | MSE en el techo (x1→x3→x10) |
|---|---|---|
| Provincial | 184.65→183.98→182.24 | 215.38→**214.04→214.04** |
| Nacional | 181.26→175.34→**175.34** | 181.26→**175.34→175.34** |

Ambos convergen a un valor estable -- confirma saturación real (no artefacto de grilla). En nacional, MSE mínimo = MSE en el techo: ningún alpha predice mejor que el modelo vacío.

In [36]:
def verificar_saturacion(X_df: pd.DataFrame, y_ser: pd.Series, factores: list[float]) -> pd.DataFrame:
    """Corre lasso_loocv_manual con distintos factor_extension y devuelve
    el MSE mínimo y el MSE en el techo de cada corrida -- si ambos se
    estabilizan al ampliar la grilla, la saturación es real, no un límite
    artificial de rango."""
    filas = []
    for factor in factores:
        r = lasso_loocv_manual(X_df, y_ser, factor_extension=factor)
        idx_min = r["mean_mse"].argmin()
        filas.append({
            "factor_extension": factor,
            "techo": r["alphas"][-1],
            "alpha_min": r["alpha_min"],
            "alpha_1se": r["alpha_1se"],
            "mse_min": r["mean_mse"][idx_min],
            "mse_en_techo": r["mean_mse"][-1],
        })
    return pd.DataFrame(filas)


for nivel in ["provincial", "nacional"]:
    X_df, y_ser = datos_final[nivel]
    print(f"--- {nivel} ---")
    print(verificar_saturacion(X_df, y_ser, factores=[1, 3, 10]))
    print()

--- provincial ---
   factor_extension      techo  alpha_min  alpha_1se     mse_min  mse_en_techo
0                 1   9.863624   4.874320   9.863624  184.650964    215.379817
1                 3  29.590871   5.450845  29.590871  183.982974    214.044218
2                10  98.636236   5.108839  98.636236  182.239166    214.044218

--- nacional ---
   factor_extension      techo  alpha_min  alpha_1se     mse_min  mse_en_techo
0                 1   9.204160   9.204160   9.204160  181.260452    181.260452
1                 3  27.612480  11.851095  27.612480  175.338629    175.338629
2                10  92.041601  12.789139  92.041601  175.338629    175.338629



### Paso 6 — Ajuste final: coeficientes y mejora sobre baseline trivial

`baseline_trivial_loocv`: MSE en LOO de predecir el promedio de los demás puntos -- piso de comparación. `mse_en_alpha`: mismo esquema de LOO que `lasso_loocv_manual`, pero para un alpha puntual (se recalcula, no se guarda en la función de CV).

**Resultado:**

| Nivel | Mejora de `alpha_min` sobre baseline | Mejora de `alpha_1se` |
|---|---|---|
| Municipal | +40.7% | +13.3% |
| Provincial | +14.0% | +0.0% |
| Nacional | +0.0% | +0.0% |

Coeficientes en `alpha_min`: `icg_pendiente_vc` sobrevive en municipal (6.41) y provincial (4.41), `reservas_pendiente_vc` solo en municipal (0.96, débil). En `alpha_1se`: solo `icg_pendiente_vc` en municipal (2.81), nada en provincial ni nacional.

In [37]:
def baseline_trivial_loocv(y_ser: pd.Series) -> float:
    """MSE en LOO de predecir, para cada punto dejado afuera, el promedio
    de los demás -- el piso de comparación: cualquier modelo que no le
    gane a esto no aporta nada."""
    n = len(y_ser)
    errores = []
    for i in range(n):
        pred = np.delete(y_ser.values, i).mean()
        errores.append((pred - y_ser.values[i]) ** 2)
    return np.mean(errores)


def mse_en_alpha(X_df: pd.DataFrame, y_ser: pd.Series, alpha: float) -> float:
    """MSE en LOO de lasso_coordinate_descent para un alpha puntual."""
    n = len(y_ser)
    errores = []
    for i in range(n):
        idx_train = [k for k in range(n) if k != i]
        X_train_df, y_train = X_df.iloc[idx_train], y_ser.iloc[idx_train].values
        X_test_df, y_test = X_df.iloc[[i]], y_ser.iloc[i]

        X_train_std, medias, desvios = estandarizar(X_train_df)
        y_train_media = y_train.mean()
        X_test_std = (X_test_df.values - medias) / desvios

        beta = lasso_coordinate_descent(X_train_std, y_train - y_train_media, alpha)
        pred = y_train_media + X_test_std @ beta
        errores.append((pred[0] - y_test) ** 2)
    return np.mean(errores)


def ajustar_final(X_df: pd.DataFrame, y_ser: pd.Series, alpha: float) -> pd.Series:
    """Ajuste final sobre todos los datos del nivel (ya no hay held-out
    acá) -- coeficientes en la escala estandarizada."""
    X_std, medias, desvios = estandarizar(X_df)
    y_centrado = y_ser.values - y_ser.values.mean()
    beta = lasso_coordinate_descent(X_std, y_centrado, alpha)
    return pd.Series(beta, index=X_df.columns)


resumen = []
coeficientes_min, coeficientes_1se = {}, {}

for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    r = resultados_cv[nivel]

    base = baseline_trivial_loocv(y_ser)
    mse_min = mse_en_alpha(X_df, y_ser, r["alpha_min"])
    mse_1se = mse_en_alpha(X_df, y_ser, r["alpha_1se"])

    resumen.append({
        "nivel": nivel,
        "baseline_mse": base,
        "mejora_alpha_min_%": 100 * (1 - mse_min / base),
        "mejora_alpha_1se_%": 100 * (1 - mse_1se / base),
    })

    coeficientes_min[nivel] = ajustar_final(X_df, y_ser, r["alpha_min"])
    coeficientes_1se[nivel] = ajustar_final(X_df, y_ser, r["alpha_1se"])

tabla_resumen = pd.DataFrame(resumen).set_index("nivel")
print(tabla_resumen)

            baseline_mse  mejora_alpha_min_%  mejora_alpha_1se_%
nivel                                                           
municipal     204.688614           40.743299           13.331969
provincial    214.044218           14.044408            0.000000
nacional      175.338629            0.000000            0.000000


In [38]:
tabla_coef_min = pd.DataFrame(coeficientes_min)
tabla_coef_min = tabla_coef_min[(tabla_coef_min != 0).any(axis=1)]
print("Coeficientes distintos de cero (alpha_min):")
tabla_coef_min

Coeficientes distintos de cero (alpha_min):


,municipal,provincial,nacional
icg_pendiente_vc,6.409051,4.412778,0.0
reservas_pendiente_vc,0.957210,0.000000,0.0


In [40]:
tabla_coef_1se = pd.DataFrame(coeficientes_1se)
tabla_coef_1se = tabla_coef_1se[(tabla_coef_1se != 0).any(axis=1)]
print("Coeficientes distintos de cero (alpha_1se):")
tabla_coef_1se

Coeficientes distintos de cero (alpha_1se):


,municipal,provincial,nacional
icg_pendiente_vc,2.81458,0.0,0.0


### Chequeo de estabilidad (leave-one-transition-out) — los tres niveles

**Municipal (alpha_1se=7.71):** Signo estable (`icg_pendiente_vc` siempre positivo), magnitud sensible a `2009_2011` y `2011_2013` (caída a 0.48/0.78 vs. 2.3-3.7 en el resto). Composición casi estable (`reservas_pendiente_vc` se activa débilmente solo al sacar `2005_2007`).

**Provincial (alpha_min=5.45 -- alpha_1se no tuvo sobrevivientes):**
- Signo estable: `icg_pendiente_vc` positivo en las 12 corridas, nunca en cero.
- Magnitud más variable que en municipal: rango 1.41 (sacando `2009_2011`) a 5.55 (sacando `2005_2007`). **`2009_2011` vuelve a ser la ventana de mayor apalancamiento, igual que en municipal** -- coincidencia entre niveles que sugiere que esa transición puntual tiene un peso real en el vínculo confianza-voto.
- Composición menos estable que en municipal: sacar `2015_2017` activa `icc_pendiente_vc` (2.02) mientras `icg` cae a 2.18 -- acá sí cambia cuál variable "aporta", no solo cuánto. `reservas_pendiente_vc`/`reservas_volatilidad_vc` aparecen de forma esporádica y débil en varias otras corridas, sin patrón consistente -- lectura: ruido, no señal.

**Nacional (alpha_min=11.85):** ninguna variable sobrevive en ninguna de las 7 corridas -- resultado nulo robusto.

**Síntesis:** `icg_pendiente_vc` es el hallazgo más sólido del ejercicio de LASSO -- signo positivo y consistente en municipal y provincial, con la particularidad de que la transición `2009_2011` reduce su magnitud en ambos niveles simultáneamente. Provincial es menos estable en composición que municipal (coherente con su menor mejora sobre baseline, 14% vs. 40.7%). Nacional no tiene ninguna señal individual bajo ningún criterio.

In [49]:
def estabilidad_seleccion(nivel: str, alpha: float) -> pd.DataFrame:
    """Reajusta el modelo final (mismo alpha ya elegido por CV) sacando
    una ventana a la vez -- para ver si la selección de variables
    depende de un punto influyente en vez de ser un patrón sostenido."""
    df = paneles[nivel]
    X_completo = df[cols_finales_municipal]
    y_completo = df["delta_v"]
    completas = X_completo.notna().all(axis=1) & y_completo.notna()

    ids = df.loc[completas, "id_transicion"].reset_index(drop=True)
    X_df, y_ser = datos_final[nivel]  # ya filtrado, mismo orden que 'ids' (misma máscara)

    n = len(y_ser)
    filas = {}
    for i in range(n):
        idx_sub = [k for k in range(n) if k != i]
        beta = ajustar_final(X_df.iloc[idx_sub], y_ser.iloc[idx_sub], alpha)
        filas[f"sin_{ids[i]}"] = beta

    return pd.DataFrame(filas).T


In [50]:
ALPHA_PARA_ESTABILIDAD = {
    "municipal": ("alpha_1se", resultados_cv["municipal"]["alpha_1se"]),
    "provincial": ("alpha_min", resultados_cv["provincial"]["alpha_min"]),
    "nacional": ("alpha_min", resultados_cv["nacional"]["alpha_min"]),
}

for nivel in NIVELES:
    criterio, alpha = ALPHA_PARA_ESTABILIDAD[nivel]
    resultado = estabilidad_seleccion(nivel, alpha)
    sobrevivientes = resultado.loc[:, (resultado != 0).any(axis=0)]

    print(f"--- {nivel} (alpha={alpha:.3f}, criterio={criterio}) ---")
    if sobrevivientes.empty:
        print("Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.\n")
    else:
        print(sobrevivientes)
        print()

--- municipal (alpha=7.712, criterio=alpha_1se) ---
                         icg_pendiente_vc  reservas_pendiente_vc
sin_municipal_2001_2003          3.280685               0.000000
sin_municipal_2003_2005          3.280341               0.000000
sin_municipal_2005_2007          2.937138               0.150497
sin_municipal_2007_2009          3.657110               0.000000
sin_municipal_2009_2011          0.478146               0.000000
sin_municipal_2011_2013          0.784155               0.000000
sin_municipal_2013_2015          2.820476               0.000000
sin_municipal_2015_2017          2.321007               0.000000
sin_municipal_2017_2019          3.589671               0.000000
sin_municipal_2019_2021          3.470692               0.000000
sin_municipal_2021_2023          3.112734               0.000000
sin_municipal_2023_2025          3.203493               0.000000

--- provincial (alpha=5.451, criterio=alpha_min) ---
                          icc_pendiente_vc  icg_p